<a href="https://colab.research.google.com/github/Nithya2405/ai-engineering-workbench/blob/main/06_supervisor_orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🧠 The Day 12 Concept: Multi-Agent Systems (MAS)

Instead of one giant, confused prompt, I split the brain:

1. **The Researcher:** Specialized only in fetching live web data.
2. **The Analyst:** Specialized in cleaning data and performing calculations.
3. **The Supervisor:** The "Project Manager" who decides who should talk when.

---

### 💻 Day 12: The Multi-Agent Supervisor

This code implements a **Router** that identifies the intent and handsoff the task to the correct specialized function.

In [9]:
!pip install groq
!pip install tavily
import os
from groq import Groq
from tavily import TavilyClient
from google.colab import userdata
import json

# 1. Setup

In [2]:
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
tavily = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

# 2. Define specialized "Worker" functions

In [3]:
def researcher_worker(query):
    print(f"🕵️ [RESEARCHER] Finding: {query}")
    return tavily.search(query=query, max_results=2)['results'][0]['content']

def analyst_worker(data):
    print(f"📊 [ANALYST] Processing data...")
    # Simulated analysis logic
    return f"Analysis complete. Key Insight: {data[:100]}..."

# 3. The Supervisor (The Brain)

In [4]:
def supervisor_orchestrator(user_request):
    print(f"👑 [SUPERVISOR] Task: {user_request}\n" + "="*40)


    # Step 1: Decision - Who should handle this?
    decision_prompt = f"""
    You are a Supervisor Agent. Based on the user request, decide if we need 'RESEARCH', 'ANALYSIS', or 'BOTH'.
    Request: {user_request}
    Response format: ONLY the word.
    """

    plan = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "system", "content": decision_prompt}]
    ).choices[0].message.content.strip()

    # Step 2: Execution based on Plan
    context = ""
    if "RESEARCH" in plan or "BOTH" in plan:
        context = researcher_worker(user_request)

    if "ANALYSIS" in plan or "BOTH" in plan:
        report = analyst_worker(context if context else user_request)
        print(f"✅ Final Result: {report}")
    else:
        print(f"✅ Final Result: {context}")

# Test the Orchestration

In [5]:
supervisor_orchestrator("Research the current NVIDIA stock price and give me a summary.")

👑 [SUPERVISOR] Task: Research the current NVIDIA stock price and give me a summary.
🕵️ [RESEARCHER] Finding: Research the current NVIDIA stock price and give me a summary.
✅ Final Result: The NVIDIA stock price today is 167.52 USD. What Stock Exchange Does NVIDIA Trade On? NVIDIA is listed and trades on the Nasdaq Stock Exchange.


#📂 Day 13: Agent Evaluation & Trace Logging
The Goal: Build a system that automatically checks if the Supervisor's final answer actually matches the user's original intent. This is called LLM-as-a-Judge.

1. The Concept: The Evaluation Loop
We will take the output from your Day 12 Supervisor and pass it to a different model (e.g., Gemini 1.5 Flash) with a rubric:

Relevance: Did it answer the actual question?

Factuality: Is the data from the researcher used correctly?

Completeness: Is it a summary or just a raw sentence?

2. The Code Logic

In [6]:
def evaluator_judge(user_request, agent_output):
    print(f"⚖️ [JUDGE] Evaluating Agent Performance...")

    evaluation_prompt = f"""
    You are an AI Quality Auditor. Grade the following Agent response based on the User Request.

    USER REQUEST: {user_request}
    AGENT RESPONSE: {agent_output}

    SCORING RUBRIC:
    1. Accuracy: Did it provide the correct data?
    2. Format: Is it a summary as requested?

    Provide a SCORE out of 10 and a 1-sentence REASON.
    Format: SCORE: [X/10] | REASON: [text]
    """

    # Using a different model to avoid "self-bias"
    judgment = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "system", "content": evaluation_prompt}]
    ).choices[0].message.content

    return judgment

# Test the Evaluation

In [7]:
request = "Research the current NVIDIA stock price and give me a summary."

# (Assuming 'result' is the output from D-12 code)

In [8]:
score = evaluator_judge(request, "The NVIDIA stock price today is 167.52 USD.")
print(f"📢 Evaluation Results:\n{score}")

⚖️ [JUDGE] Evaluating Agent Performance...
📢 Evaluation Results:
SCORE: 6/10 | REASON: The agent provided a potentially accurate stock price, but failed to deliver a summary as requested, instead offering only a brief statement with the current price.
